# The k-means assign kernel: what changed, and what holds up

Run top to bottom on an **A100** (Runtime > Change runtime type). It takes about 15 minutes.

Claims you can check here:
1. **iter06** (phase 1) is about 1.36× faster than the original on real keys and passes the strict correctness test
   on real and held-out keys. A third agent pass seeded with iter06 found nothing faster.
2. The **int8** kernel is 1.74× faster on synthetic keys and passes phase 1's test there, but fails the strict test.
3. **Phase 2's** 2.4× came from key files that skipped a step of Double-P's pipeline; on real keys it is slower
   than iter06.
4. The harness: timing in one process, the repeat-call check, and clock drift between processes.

Needs, in Google Drive under `MyDrive/kernel-agent/`: `centered/real_keys_gate.pt`, `centered/real_keys_heldout.pt`
(made by `test.ipynb`), `runs/20261026-004515/iter06.py`, `runs/20261026-004515/best_kernel.py` (int8) and
`runs/20261026-033411/best_kernel.py` (phase 2).

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/kernel-agent"
copies = {"k_iter06.py": f"{D}/runs/20261026-004515/iter06.py",
          "k_int8.py": f"{D}/runs/20261026-004515/best_kernel.py",
          "k_phase2.py": f"{D}/runs/20261026-033411/best_kernel.py"}
for p in [f"{D}/centered/real_keys_gate.pt", f"{D}/centered/real_keys_heldout.pt", *copies.values()]:
    print("ok     " if os.path.exists(p) else "MISSING", p)
for dst, src in copies.items():
    if os.path.exists(src):
        shutil.copy(src, f"/content/{dst}")

## 1. What the kernel does

One step of k-means on attention keys. For each key (128 numbers), find the centroid with the highest dot product,
then add the key into that centroid's running sum `s` and add 1 to its count `cnt`. The next step divides `s` by `cnt`
to get the new centroids.

Shapes: 8 rows, each one (layer, head) of the model; 32,768 keys per row; 2,048 centroids; 128 dims. Each Triton
program takes 128 keys of one row and loops over the centroids 32 at a time. It scores them on tensor cores with
`tl.dot`, keeps the best centroid per key, and finishes with the atomic adds.

| Keys | What they are | Used for |
|---|---|---|
| synthetic | random Gaussian keys, random unit-length centroids | phase 1's only test and timing data |
| gate keys | real Qwen2.5-1.5B keys from one 32,836-token prompt (Pride and Prejudice from its start), prepared the way Double-P clusters them (below); 8 (layer, head) rows (layers 0, 3, 7, …, 27); centroids from Double-P's segment k-means | the third pass's test and timing data (the agent saw these) |
| held-out keys | same method, a prompt from book token 80,000, other (layer, head) pairs (layers 0, 2, 6, …, 26) | checked only after the run (the agent never saw them) |

**How Double-P prepares keys, and what the earlier files got wrong.** Double-P clusters keys once, at prefill
(`RetroClusterIndex.build` in `sparse_attention/index/cluster/retro_cluster.py`). It leaves out the first 4 tokens
(attention sinks) and the last 64 (recent window), which it keeps exact, and it subtracts each head's mean key before
k-means. The key files phases 1 and 2 used skipped both steps. That mattered on layer 0: Qwen2.5's key projection adds
a bias, and in layer 0 that bias is as long as the keys themselves (|bias| ≈ 821 against a median |key| ≈ 821 in
head 0), so raw layer-0 keys all point nearly the same way. Up to 99% of them landed in one cluster, and all their
atomic adds hit the same addresses. The original kernel was very slow on that input, which inflated every layer-0
speedup in the earlier runs. Subtracting the mean removes the bias. In the files used here the biggest cluster holds
0.1–0.3% of keys in every row, layer 0 included.

The original kernel is below; read this one. The next cells are helpers you can run without reading.

In [ ]:
%%writefile k_original.py
"""The original assign kernel from Double-P (sparse_attention/index/cluster/kmeans.py, _triton_assign_kernel).

One program handles BLOCK_N keys of one batch row: score them against every centroid with tensor cores, keep the best
centroid per key, then add each key into its centroid's running sum (s) and count (cnt) with atomics.
"""

import triton
import triton.language as tl


@triton.jit
def _assign(
    K, X, S, C, M,
    stride_kz: tl.constexpr, stride_kn: tl.constexpr, stride_kd: tl.constexpr,
    stride_xz: tl.constexpr, stride_xk: tl.constexpr, stride_xd: tl.constexpr,
    stride_sz: tl.constexpr, stride_sk: tl.constexpr, stride_sd: tl.constexpr,
    stride_cz: tl.constexpr, stride_ck: tl.constexpr,
    stride_mz: tl.constexpr, stride_mn: tl.constexpr,
    num_tokens: tl.constexpr, num_centroids: tl.constexpr,
    BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr, BLOCK_D: tl.constexpr,
):
# PROLOGUE of Tile processing
    start_n = tl.program_id(0) * BLOCK_N
    batch_idx = tl.program_id(1)
    if start_n >= num_tokens:
        return
    offs_n = start_n + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    offs_d = tl.arange(0, BLOCK_D)
    n_mask = offs_n < num_tokens
    k_ptrs = K + batch_idx * stride_kz + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
    x_ptrs = X + batch_idx * stride_xz + offs_k[None, :] * stride_xk + offs_d[:, None] * stride_xd
    s_ptrs = S + batch_idx * stride_sz + offs_d[None, :] * stride_sd
    c_ptrs = C + batch_idx * stride_cz
    m_ptrs = M + batch_idx * stride_mz + offs_n * stride_mn

    # start of Main Loop
    k = tl.load(k_ptrs, mask=n_mask[:, None], other=0.0)  # [BLOCK_N, D] keys, loaded once
    max_val = tl.zeros([BLOCK_N], dtype=tl.float32) - float("inf")
    max_idx = tl.zeros([BLOCK_N], dtype=tl.int32) 

    for start_k in tl.range(0, num_centroids, BLOCK_K):
        x = tl.load(x_ptrs)  # [D, BLOCK_K] centroids
        ip = tl.dot(k, x).to(tl.float32)  # [k space, centroid space] block K is centroids, block N is k
        tmp_max_val, tmp_max_idx = tl.max(ip, axis=1, return_indices=True) # get max centroid idx, with max score value
        tmp_max_idx += start_k # increment tmp_max_ids to reflect the block offset in the centroid space
        max_idx = tl.where(tmp_max_val > max_val, tmp_max_idx, max_idx) # elemenwise if condition for each 1D value of K, update max_idx if tmp_max_val is greater
        max_val = tl.maximum(tmp_max_val, max_val) # elementwise maximum of the current max and the new max values
        x_ptrs += BLOCK_K * stride_xk # move the pointer to the next block of centroids
    
    # EPILOGUE of Tile processing
    tl.store(m_ptrs, max_idx, mask=n_mask) #[BLOCK_N] max_idx
    # s[label[n], :] += k[n, :] for the whole [BLOCK_N, D] tile in one statement. The atomics use the register layout
    # Triton chose for loading k and feeding the tensor cores, which is not arranged for atomics (see the notebook).
    # [block_n, dim] atomic add
    # actually the matrix is [k projection, centroid space] and they are memory addresses of a centroid vector to 
    # write a tile of k into.
    # each row in that k tile corresponds to a centroid address
    # we can have many same centroid addresses in the k tile
    # tl.atomic_add is read, modify, write where modify means += and read,modify,write means a single atomic operation. it cannot be interrupted mid operation.  This avoids race conditions and concurrency issues.
    # it is at the cost of performance, as now operations are serialized and cannot be overlapped.
    # internally each element of the tile is processed one at a time.  This goes from O(1) elementwise ops to O(BLOCK_N * D) atomic operations.
    # s_ptrs just gets you in the ballpark
    # max_idx contains the index of the centroid with the maximum score for each row in the k tile
    # strid_sk gives you row stride for each index in the k tile ( SO WE CAN FILL IN THE ELEMENTS OF EACH VECTOR IN THE ROW)

    # notice we are passing 2D matrix to atomic_add
    tl.atomic_add(s_ptrs + max_idx[:, None] * stride_sk, k.to(tl.float32), mask=n_mask[:, None], sem="relaxed")
    # cnt[label[n]] += 1
    # this is adding count vectors for later component wise scalar division [1,1,1,...] 
    # shape [BLOCK_N]
    tl.atomic_add(c_ptrs + max_idx * stride_ck, tl.zeros_like(max_idx) + 1, mask=n_mask, sem="relaxed") 


def launch(data, cent, s, cnt, m):
    """data [B, N, D] bf16, cent [B, C, D] bf16 (unit norm), s [B, C, D] fp32 and cnt [B, C] int32 zeroed, m [B, N] int32."""
    b, n, d = data.shape
    c = cent.shape[1]
    _assign[(triton.cdiv(n, 128), b, 1)](
        data, cent, s, cnt, m,
        *data.stride(), *cent.stride(), *s.stride(), *cnt.stride(), *m.stride(),
        n, c,
        BLOCK_N=128, BLOCK_K=32, BLOCK_D=d, num_warps=4, num_stages=2,  # the config kmeans.py uses
    )

In [ ]:
%%writefile k_ablation.py
"""Ablation: the original kernel with iter06's two changes switched on one at a time.

Written for this notebook (not by the agent), from the agent's code, with the original's tile sizes (128 keys x 32
centroids, 4 warps, 2 stages) so that only the two changes differ:

    PACKED   False: argmax as in the original (max + index search + select per tile)
             True:  pack the centroid index into the low bits of the fp32 score, so the argmax is a single max
    EPI      0: atomics straight from the key tile, as in the original
             1: re-read the keys in a flat layout, one key row after another (compiler can see the addresses are
                contiguous)
             2: same as 1, but the column offset is multiplied by a runtime 1 (stride_one) that the compiler cannot
                see through, as the agent did
"""

import triton
import triton.language as tl


@triton.jit(do_not_specialize=["stride_one"])
def _assign(
    K, X, S, CNT, M, N, C, stride_one,
    BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr, D: tl.constexpr,
    MASK_HI: tl.constexpr, MASK_LO: tl.constexpr,
    PACKED: tl.constexpr, EPI: tl.constexpr, EPI_ROWS: tl.constexpr,
):
    pid_n = tl.program_id(0)
    b = tl.program_id(1).to(tl.int64)
    start_n = pid_n * BLOCK_N
    offs_n = start_n + tl.arange(0, BLOCK_N)
    offs_d = tl.arange(0, D)
    offs_k = tl.arange(0, BLOCK_K)
    n_mask = offs_n < N
    k = tl.load(K + b * N * D + offs_n[:, None] * D + offs_d[None, :], mask=n_mask[:, None], other=0.0)
    x_ptrs = X + b * C * D + offs_k[None, :] * D + offs_d[:, None]

    if PACKED:
        # score bits with the low IDX_BITS cleared, OR the centroid index: comparing the packed floats compares
        # scores (to 12 fraction bits for 2048 centroids) and carries the winner's index along for free
        best = tl.full([BLOCK_N, BLOCK_K], float("-inf"), tl.float32)
        for start in range(0, C, BLOCK_K):
            ip = tl.dot(k, tl.load(x_ptrs))
            bits = ip.to(tl.int32, bitcast=True)
            idx = start + offs_k
            best = tl.maximum(best, ((bits & MASK_HI) | idx[None, :]).to(tl.float32, bitcast=True))
            x_ptrs += BLOCK_K * D
        max_idx = tl.max(best, axis=1).to(tl.int32, bitcast=True) & MASK_LO
    else:
        max_val = tl.full([BLOCK_N], float("-inf"), tl.float32)
        max_idx = tl.zeros([BLOCK_N], dtype=tl.int32)
        for start in range(0, C, BLOCK_K):
            ip = tl.dot(k, tl.load(x_ptrs))
            v, i = tl.max(ip, axis=1, return_indices=True)
            max_idx = tl.where(v > max_val, i + start, max_idx)
            max_val = tl.maximum(v, max_val)
            x_ptrs += BLOCK_K * D

    m_base = M + b * N
    tl.store(m_base + offs_n, max_idx, mask=n_mask)
    tl.atomic_add(CNT + b * C + max_idx, tl.zeros_like(max_idx) + 1, mask=n_mask, sem="relaxed")
    if EPI == 0:
        tl.atomic_add(S + b * C * D + max_idx[:, None] * D + offs_d[None, :], k.to(tl.float32),
                      mask=n_mask[:, None], sem="relaxed")
    else:
        tl.debug_barrier()  # the labels stored above must be visible to every thread before they are re-read
        ii = tl.arange(0, EPI_ROWS * D)
        r_loc = ii // D  # which key row
        cc = ii % D  # which column: consecutive threads get consecutive columns of the same row
        if EPI == 2:
            cc = cc * stride_one
        for r0 in range(0, BLOCK_N, EPI_ROWS):
            rr = start_n + r0 + r_loc
            rmask = rr < N
            lab = tl.load(m_base + rr, mask=rmask, other=0)
            kv = tl.load(K + b * N * D + rr * D + cc, mask=rmask, other=0.0)  # re-read keys (from L2)
            tl.atomic_add(S + b * C * D + lab * D + cc, kv.to(tl.float32), mask=rmask, sem="relaxed")


def variant(packed, epi):
    def launch(data, cent, s, cnt, m):
        b, n, d = data.shape
        c = cent.shape[1]
        bits = max(1, (c - 1).bit_length())
        _assign[(triton.cdiv(n, 128), b, 1)](
            data, cent, s, cnt, m, n, c, 1,
            BLOCK_N=128, BLOCK_K=32, D=d, MASK_HI=-(1 << bits), MASK_LO=(1 << bits) - 1,
            PACKED=packed, EPI=epi, EPI_ROWS=32, num_warps=4, num_stages=2,
        )
    return launch

In [ ]:
%%writefile lab.py
"""Helpers for the notebook: inputs, kernels, correctness checks and same-process timing."""

import importlib.util
import statistics
from pathlib import Path

import torch
import triton

DEV = "cuda"
HERE = Path(__file__).resolve().parent
DRIVE = "/content/drive/MyDrive/kernel-agent"
GATE, HELDOUT = f"{DRIVE}/centered/real_keys_gate.pt", f"{DRIVE}/centered/real_keys_heldout.pt"
ABS_TOL = 0.05  # phase-1 rule: chosen centroid's score within 0.05 of the best (absolute)
REL_TOL = 2.0 ** -12  # phase-2 rule: within 2^-12 x |key| of the best, i.e. a cosine gap (centroids are unit norm)
torch.backends.cuda.matmul.allow_tf32 = False  # the reference must be real fp32

FILES = {"original": "k_original.py", "iter06": "k_iter06.py", "int8": "k_int8.py", "phase2": "k_phase2.py"}
ABLATION = {  # name: (PACKED, EPI), see k_ablation.py
    "plain argmax, tile atomics": (False, 0),
    "packed argmax, tile atomics": (True, 0),
    "plain argmax, flat atomics": (False, 2),
    "packed, flat (stride visible)": (True, 1),
    "packed, flat (iter06 style)": (True, 2),
}


def load_kernel(path):
    spec = importlib.util.spec_from_file_location(Path(path).stem, str(path))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.launch


def int8_ncand(n):
    """The int8 kernel limited to its configs that rescore n candidates (its own tuner picks among 2, 3 and 4)."""
    spec = importlib.util.spec_from_file_location(f"k_int8_x{n}", str(HERE / FILES["int8"]))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    mod._CONFIGS = [cfg for cfg in mod._CONFIGS if cfg[0] == "i8" and cfg[-1] == n]
    assert mod._CONFIGS, f"k_int8.py has no int8 config with {n} candidates"

    def launch(data, cent, s, cnt, m):
        mod.launch(data, cent, s, cnt, m)
        # if none of them compiles, its tuner falls back to a bf16 config
        assert all(cfg[0] == "i8" for cfg in mod._CACHE.values()), f"no int8 config with {n} candidates ran"
    return launch


def kernels():
    return {**{name: load_kernel(HERE / f) for name, f in FILES.items()}, "int8 x4": int8_ncand(4)}


def ablation():
    spec = importlib.util.spec_from_file_location("k_ablation", str(HERE / "k_ablation.py"))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return {name: mod.variant(*flags) for name, flags in ABLATION.items()}


def synthetic(b=8, n=32768, c=2048, d=128, seed=0):
    """Random Gaussian keys and random unit centroids: no structure at all."""
    g = torch.Generator(device=DEV).manual_seed(seed)
    keys = torch.randn(b, n, d, device=DEV, generator=g).bfloat16()
    cent = torch.randn(b, c, d, device=DEV, generator=g)
    rows = [("row", i) for i in range(b)]
    return keys, (cent / cent.norm(dim=-1, keepdim=True)).bfloat16(), rows


def real(path):
    """Real Qwen2.5-1.5B keys [8, 32768, 128] bf16, prepared as Double-P clusters them (sink and window tokens dropped,
    mean subtracted), their segment k-means centroids [8, 2048, 128], (layer, head) per row."""
    d = torch.load(path, map_location="cpu")
    return d["keys"].to(DEV), d["cent"].to(DEV), [tuple(r) for r in d["rows"]]


def one_row(keys, cent, i):
    """Row i copied into every batch slot: same shape (so each kernel keeps its config), one (layer, head)."""
    b = keys.shape[0]
    return keys[i:i + 1].expand(b, -1, -1).contiguous(), cent[i:i + 1].expand(b, -1, -1).contiguous()


def inputs(which):
    """synthetic | normal (gate row layer 3 head 1), each [8, 32768, 128]."""
    if which == "synthetic":
        return synthetic()[:2]
    keys, cent, _ = real(GATE)
    return one_row(keys, cent, {"normal": 1}[which])


def alloc(keys, cent):
    b, n, d = keys.shape
    c = cent.shape[1]
    return (torch.zeros(b, c, d, device=DEV), torch.zeros(b, c, device=DEV, dtype=torch.int32),
            torch.full((b, n), -1, device=DEV, dtype=torch.int32))


def run(launch, keys, cent):
    s, cnt, m = alloc(keys, cent)
    launch(keys, cent, s, cnt, m)
    torch.cuda.synchronize()
    return s, cnt, m


def check(keys, cent, s, cnt, m):
    """Per row, against exact fp32 math: label agreement, worst score gap (absolute, and divided by |key|), and whether
    the sums and counts are exactly what the kernel's own labels imply."""
    out = []
    for i in range(keys.shape[0]):
        k, x, lab = keys[i].float(), cent[i].float(), m[i].long()
        if int(lab.min()) < 0 or int(lab.max()) >= x.shape[0]:
            out.append({"agree": 0.0, "abs_gap": float("inf"), "cos_gap": float("inf"), "sums_ok": False})
            continue
        sc = k @ x.T  # [N, C] exact scores
        gap = sc.max(1).values - sc.gather(1, lab[:, None]).squeeze(1)
        s_ref = torch.zeros(x.shape, device=DEV, dtype=torch.float64).index_add_(0, lab, k.double())
        cnt_ref = torch.bincount(lab, minlength=x.shape[0])
        sum_err = ((s[i].double() - s_ref).abs().max() / s_ref.abs().max()).item()
        out.append({"agree": (lab == sc.argmax(1)).double().mean().item(), "abs_gap": gap.max().item(),
                    "cos_gap": (gap / k.norm(dim=1).clamp_min(1e-30)).max().item(),
                    "sums_ok": bool(torch.equal(cnt[i].long(), cnt_ref)) and sum_err < 1e-5})
    return out


def rule1(r):
    return r["sums_ok"] and r["abs_gap"] < ABS_TOL


def rule2(r):
    return r["sums_ok"] and r["cos_gap"] <= REL_TOL


def compare(kernels, keys, cent, rounds=9):
    """Time every kernel in this one process, in rounds: each round times each kernel once (order rotated), and a
    kernel's score is the median over rounds of (original time / its time), so clock drift cancels. Each call zeroes
    s and cnt first, as a real k-means iteration does; zeroing alone is timed too and subtracted.
    Returns {name: (median kernel ms, median speedup vs "original", (lowest, highest) per-round speedup)}."""
    s, cnt, m = alloc(keys, cent)

    def make(launch):
        def f():
            s.zero_()
            cnt.zero_()
            launch(keys, cent, s, cnt, m)
        return f

    fns = {name: make(launch) for name, launch in kernels.items()}
    fns["_zero"] = lambda: (s.zero_(), cnt.zero_())
    for f in fns.values():  # compile (and autotune) before timing
        f()
    triton.testing.do_bench(next(iter(fns.values())), warmup=500, rep=500)  # keep the GPU busy so clocks settle
    names = list(fns)
    raw = {k: [] for k in names}
    for r in range(rounds):
        for k in names[r % len(names):] + names[:r % len(names)]:
            raw[k].append(triton.testing.do_bench(fns[k], warmup=25, rep=100))
    zero = raw.pop("_zero")
    t = {k: [a - z for a, z in zip(v, zero)] for k, v in raw.items()}
    out = {}
    for k, ts in t.items():
        ratios = sorted(o / x for o, x in zip(t["original"], ts))
        out[k] = (statistics.median(ts), statistics.median(ratios), (ratios[0], ratios[-1]))
    return out


def show(res):
    for name, (ms, x, (lo, hi)) in res.items():
        print(f"  {name:32s} {ms:.3f} ms  {x:.3f}x vs original (rounds {lo:.3f}-{hi:.3f})")

In [ ]:
%%writefile prof.py
"""One measured launch of a kernel, for Nsight Compute. Run under: ncu --profile-from-start off ... python prof.py ...

    python prof.py KERNEL INPUT     # KERNEL: original | iter06 | int8 | phase2 | an ablation name
                                    # INPUT: synthetic | normal
"""

import sys

import torch

import lab

name, which = sys.argv[1], sys.argv[2]
launch = {**lab.kernels(), **lab.ablation()}[name]
keys, cent = lab.inputs(which)
s, cnt, m = lab.alloc(keys, cent)
launch(keys, cent, s, cnt, m)  # compile, autotune and warm up outside the profiled range
torch.cuda.synchronize()
s.zero_()
cnt.zero_()
torch.cuda.synchronize()
torch.cuda.profiler.start()  # ncu profiles only the kernels launched between start and stop
launch(keys, cent, s, cnt, m)
torch.cuda.synchronize()
torch.cuda.profiler.stop()
print(f"profiled {name} on {which}")

In [ ]:
%%writefile drift.py
"""Time the original kernel on synthetic keys in this fresh process, the way the first agent runs timed each candidate."""

import statistics

import triton

import lab

keys, cent, _ = lab.synthetic()
s, cnt, m = lab.alloc(keys, cent)
launch = lab.load_kernel(lab.HERE / "k_original.py")


def f():
    s.zero_()
    cnt.zero_()
    launch(keys, cent, s, cnt, m)


f()
print(statistics.median(triton.testing.do_bench(f, warmup=25, rep=100) for _ in range(3)))

## 2. The four kernels and what changed

### Atomic adds
After it chooses a centroid for every key, the kernel does `s[label] += key` (128 floats) and `cnt[label] += 1`.
Thousands of threads update `s` at the same time, so these are **atomic** adds: the L2 cache does each
read-add-write as one indivisible step, so no update is lost. `sem="relaxed"` promises nothing beyond that, so the
order of the additions (and the last bits of `s`) varies from run to run.

Two things make atomic adds slow:
- **Coalescing.** A warp is 32 threads that issue one instruction together, and memory works in 32-byte sectors
  (8 floats). If the 32 threads touch 32 neighbouring floats, the instruction needs 4 sectors. If they're spread out,
  it can need up to 32, which is up to 8× the L2 work for the same data. *Coalesced* means neighbouring threads
  touch neighbouring addresses.
- **Collisions.** Two adds to the same float can't happen at once; the L2 does them one after the other. When many
  keys pick the same centroid at the same time, their adds queue up on the same 128 floats. That is what happened on
  the uncentered layer-0 rows. On real keys it happens a little: neighbouring tokens often share a centroid, so a
  tile of 128 real keys picks only 9–23 distinct centroids, against about 124 for random keys.

### original
- **Argmax:** every loop iteration does `tl.max(..., return_indices=True)`, then a compare and a select.
- **Atomics:** one `tl.atomic_add` over the whole [128 keys × 128 dims] tile, in the register layout Triton chose
  for loading `k` and feeding the tensor cores. That layout is arranged for loads, not atomics. The atomic sectors
  per request in section 6 shows how spread out the atomics really are.

### iter06 (phase 1; the kernel the bullet is about)
1. **Packed argmax.** It clears the low 11 bits of each fp32 score and writes the centroid index there (2,048
   centroids need 11 bits). Clearing the bits rounds the score down a little, and the index changes it by less than
   that rounding step. So one `max` over the packed values compares the rounded scores and carries the winner's
   index in its low bits, in place of a max, an index search and a select.
   **The cost:** scores are compared with 12 fraction bits instead of 23. It can therefore pick a centroid whose
   score is below the best by up to 2^-12 of the score, which is at most 2^-12 × |key| because centroids have
   length 1. That bound is exactly phase 2's tolerance, which is 16× finer than bf16, the precision the keys are
   stored in. My CPU emulation of the packing gives a worst case of 1.9e-4 on these keys (gate layer 0), against the
   bound of 2.44e-4.
2. **Coalesced atomics.** After it stores the labels, it re-reads the keys (still in L2) as one flat list:
   `ii = tl.arange(0, EPI_ROWS * D)`, with row `ii // D` and column `ii % D`. Neighbouring threads therefore add
   neighbouring columns of one key into `s[label]`. In the agent's kernels the column offset is also multiplied by
   `stride_one`, a runtime 1 marked `do_not_specialize` so the compiler can't see that it is 1. The cell below shows
   whether iter06 does this. My guess is that it stops the compiler from handing each thread several neighbouring
   values, which suits loads but spreads out the atomics. The ablation in section 4 tests this guess.

### int8 (phase 1's fastest; fails on real keys)
- **Scoring:** keys and centroids are scored in int8 on tensor cores (on A100: 624 int8 TOPS vs 312 bf16 TFLOPS).
  Each key is scaled so its largest entry becomes 127, and all centroids share one scale.
- **Rescoring:** it keeps the 2 best approximate candidates per key, rescores those 2 exactly in fp32, and keeps
  the better one.
- **Failure:** it picks wrong whenever the true best centroid isn't among the 2 candidates. Its own tuner can also
  rescore 3 or 4, but 2 was fastest.
- **Where it fails:** on random keys the best few centroids score almost the same, so int8 rounding reorders them;
  my CPU emulation gives a cos gap of about 2e-3 with 2 candidates and 1e-3 even with 4. On real keys each key sits
  close to one k-means centroid, so the margins are wider. With 2 candidates it is exact on 13 of the 16 rows and
  fails on both layer-0 rows and layer 11 head 1 (cos gap up to 3e-3, 99.97% agreement); with 4 it passes all 16.
  `int8 x4` below is the same kernel limited to 4 candidates, to test that on the GPU. (On the old uncentered files
  int8 failed far worse: the bias made a few entries of each layer-0 key huge, so half the others rounded to 0.)

### phase2 (phase 2's best)
- **Persistent kernel:** it launches a fixed number of programs (about 3 per SM) that loop over the work.
- **Split tiles:** one tile's centroid loop can be split across programs. They merge their best (score, index)
  with a 64-bit `atomic_max`, and the last one to finish does the atomic adds.
- **Precision:** it packs only the 6-bit tile number into the score, keeping 17 fraction bits. Bound
  2^-17 ≈ 7.6e-6.
- **What it was tuned on:** the old uncentered gate keys, where the original kernel was very slow on the layer-0
  row. Its speedup there came from that row.

### Third pass (centered keys)
The agent loop was run again on the centered gate keys, starting from iter06 and told about phase 2's 6-bit packing.
In 6 iterations it found nothing that was both correct and faster than iter06, and stopped at its patience limit.
So iter06 is still the kernel to report.

The cell below prints the lines where these changes live; `!cat k_iter06.py` shows a whole file.

In [ ]:
import re
patterns = {"original": r"atomic_add|tl\.max\(|tl\.where",
            "iter06": r"atomic_add|MASK_HI|MASK_LO|stride_one|// D|% D|num_warps",
            "int8": r"atomic_add|tl\.int8|out_dtype|NCAND|127\.0",
            "phase2": r"atomic_add|atomic_max|MASK_HI|while it|maxnreg"}
for name, pat in patterns.items():
    print(f"===== k_{name}.py")
    for i, line in enumerate(open(f"k_{name}.py"), 1):
        if re.search(pat, line):
            print(f"{i:4d}  {line.rstrip()}")

## 3. Correctness

Each kernel's labels are compared with exact fp32 math on every key:
- **agree:** share of keys that get the same centroid as exact math. Near-ties can flip without harm.
- **abs gap:** worst (best score − chosen score).
- **cos gap:** the same gap divided by |key|. Centroids have length 1, so this is a gap in cosine similarity.
- **sums and counts:** `s` and `cnt` must match the kernel's own labels exactly.

Two rules:
- **Phase-1 rule** (phase 1 checked only synthetic keys): abs gap < 0.05.
- **Phase-2 rule:** cos gap ≤ 2^-12 ≈ 2.44e-4. It doesn't depend on key length. Centered real keys range from
  |key| ≈ 11 to ≈ 51 (layer 0 is longest), and the old uncentered layer-0 keys were ≈ 820, so a fixed absolute gap
  is strict on long keys and loose on short ones.

The agent's harness applies the phase-2 rule to synthetic keys as well as real ones.

In [ ]:
import statistics, lab
K = lab.kernels()
sets = {"synthetic": lab.synthetic(), "gate keys": lab.real(lab.GATE), "held-out keys": lab.real(lab.HELDOUT)}
label = lambda lh: f"L{lh[0]}h{lh[1]}" if lh[0] != "row" else f"row{lh[1]}"
for set_name, (keys, cent, rows) in sets.items():
    print(set_name)
    for name, launch in K.items():
        res = lab.check(keys, cent, *lab.run(launch, keys, cent))
        f1 = [label(lh) for lh, r in zip(rows, res) if not lab.rule1(r)]
        f2 = [label(lh) for lh, r in zip(rows, res) if not lab.rule2(r)]
        print(f"  {name:8s} phase-1 rule {'PASS' if not f1 else 'FAIL'} | phase-2 rule {'PASS' if not f2 else 'FAIL'} | "
              f"min agree {min(r['agree'] for r in res):.5f} | worst abs gap {max(r['abs_gap'] for r in res):.1e} | "
              f"worst cos gap {max(r['cos_gap'] for r in res):.1e} | failing rows: rule 1 {f1 or '-'}, rule 2 {f2 or '-'}")

What to look for (predictions from my CPU emulation):
- **int8** should pass the phase-1 rule on synthetic keys, which is how it got through phase 1, and fail the
  phase-2 rule there (cos gap ≈ 2e-3). So phase 1's loose absolute tolerance mattered as much as its synthetic
  data. On real keys it should fail both rules on the two layer-0 rows (abs gaps 0.06 and 0.13) and the phase-2
  rule on layer 11 head 1.
- **int8 x4** should pass the phase-2 rule on all real rows and still fail it on synthetic keys. The harness would
  reject it for the synthetic failure alone. Random keys are a harsher test than real ones here, because real keys
  sit near their centroids.
- **iter06** should pass the phase-2 rule everywhere, with a worst cos gap near 1.9e-4.
- On the old uncentered files, iter06 failed the phase-1 rule on layer 0 (abs gap 0.06, because |key| ≈ 820) even
  though its cos gap was only 7e-5. That mismatch is why phase 2 switched rules.

## 4. Speed

GPU clocks drift. In the first runs, timing two kernels in different processes and comparing the numbers was off by
up to ~18% (section 5 lets you test that). So here every kernel is timed in the same process, in alternating rounds,
and scored by the median per-round ratio to the original. Times are kernel-only: zeroing `s` and `cnt` is timed
separately and subtracted.

### Whole batches (8 rows at once)

In [ ]:
for set_name, (keys, cent, _) in sets.items():
    print(set_name)
    lab.show(lab.compare(K, keys, cent))

Earlier results: int8 1.74× on synthetic keys. On the old uncentered gate keys, phase2 was 2.36× and iter06 2.13×,
because the original was slow on the layer-0 row. Every row here is an ordinary head, so expect iter06 near 1.36×
and phase2 below it. `int8 x4` shows what the extra candidates cost.

### One (layer, head) at a time
Each row is copied into all 8 batch slots, so the shape (and each kernel's chosen config) stays the same. On the old
files' ordinary rows, iter06 was 1.35–1.38× and phase2 1.25–1.28×.

In [ ]:
pick = {k: K[k] for k in ["original", "iter06", "phase2"]}
table = []
for set_name in ["gate keys", "held-out keys"]:
    keys, cent, rows = sets[set_name]
    for i, (layer, head) in enumerate(rows):
        res = lab.compare(pick, *lab.one_row(keys, cent, i), rounds=5)
        table.append(res)
        print(f"L{layer:2d}h{head}  original {res['original'][0]:.3f} ms | iter06 {res['iter06'][0]:.3f} ms "
              f"{res['iter06'][1]:.2f}x | phase2 {res['phase2'][0]:.3f} ms {res['phase2'][1]:.2f}x")
print()
for k in ["iter06", "phase2"]:
    xs = [r[k][1] for r in table]
    print(f"{k}: {len(xs)} rows {min(xs):.2f}-{max(xs):.2f}x (median {statistics.median(xs):.2f}x)")

### Ablation: what each change buys
The five variants in `k_ablation.py` use the original's tile sizes, so only the two changes differ:

| Variant | Argmax | Atomic adds |
|---|---|---|
| 1 | as in the original | straight from the key tile, as in the original |
| 2 | packed | from the key tile |
| 3 | as in the original | flat re-read (iter06 style) |
| 4 | packed | flat re-read, with the compiler able to see the stride |
| 5 | packed | flat re-read, iter06 style |

Variant 1 should time close to the original. If it doesn't, the gap comes from details such as runtime sizes, so
compare the variants with each other. iter06 is included for reference; it may use different tile sizes.

In [ ]:
A = lab.ablation()
keys, cent, rows = sets["gate keys"]
print("correctness on gate keys (phase-2 rule)")
for name, launch in A.items():
    res = lab.check(keys, cent, *lab.run(launch, keys, cent))
    print(f"  {name:32s} {'PASS' if all(map(lab.rule2, res)) else 'FAIL'}  worst cos gap {max(r['cos_gap'] for r in res):.1e}")
for which in ["synthetic", "normal"]:
    print(f"\n{which}")
    lab.show(lab.compare({"original": K["original"], **A, "iter06": K["iter06"]}, *lab.inputs(which), rounds=7))

## 5. Harness checks

**Repeat-call check.** The timing loop calls a kernel hundreds of times, but only the first call is checked. A kernel
that skips the work on later calls would pass the check and time near zero. The harness therefore re-checks one call
made after timing. Below is a deliberately lazy wrapper around iter06.

In [ ]:
iter06, seen = K["iter06"], set()
def lazy(data, cent, s, cnt, m):
    """Does the work on the first call for each input, then returns immediately."""
    if data.data_ptr() in seen:
        return
    seen.add(data.data_ptr())
    iter06(data, cent, s, cnt, m)

keys, cent, _ = sets["synthetic"]
res = lab.check(keys, cent, *lab.run(lazy, keys, cent))
print("first call, checked:   ", "PASS" if all(map(lab.rule2, res)) else "FAIL")
t = lab.compare({"original": K["original"], "lazy": lazy}, keys, cent, rounds=3)
print(f"timed: lazy {t['lazy'][0] * 1e3:.0f} us per call vs original {t['original'][0] * 1e3:.0f} us")
res = lab.check(keys, cent, *lab.run(lazy, keys, cent))
print("one more call, checked:", "PASS" if all(map(lab.rule2, res)) else "FAIL  <- caught by the repeat-call check")

**Drift between processes.** Each run below is a fresh process timing the same original kernel, the way the first
agent runs timed each candidate in its own process. The spread is the error you'd get comparing numbers from
different processes. Drift depends on GPU temperature and boost state, so you may see less than the ~18% seen
earlier. If you do, don't quote 18% unless you can find it in the earlier logs.

In [ ]:
import subprocess
def one():
    r = subprocess.run(["python", "drift.py"], capture_output=True, text=True, cwd="/content")
    if r.returncode:
        raise RuntimeError(r.stderr[-1500:])
    return float(r.stdout.split()[-1])
ts = [one() for _ in range(6)]
print("original kernel, one fresh process each:", ", ".join(f"{t:.3f}" for t in ts), "ms")
print(f"spread: slowest / fastest = {max(ts) / min(ts):.3f}")

## 6. Nsight Compute: why each kernel is as fast as it is

`prof.py` makes one launch between `torch.cuda.profiler.start()` and `stop()`, and `ncu --profile-from-start off`
profiles only that launch. int8 launches three kernels (two small ones quantize the centroids); the table shows the
main one. Nsight replays each kernel many times and locks clocks at base frequency, so its durations are longer than
section 4's. Compare kernels with each other, not with section 4.

How to read the rows:
- **Compute / memory throughput %** ("speed of light"): how close the busiest compute unit and the busiest memory
  unit came to their peaks. If neither is near ~80%, hardware throughput isn't the limit; latency is, and warps sit
  waiting.
- **Tensor pipe %:** how busy the tensor cores are (bf16 for original, iter06 and phase2; int8 for int8).
- **DRAM %:** tiny for all of these. Keys and centroids are reused from cache, so memory bandwidth isn't the problem.
- **Registers, blocks/SM limits, achieved occupancy:** how many warps fit on an SM at once. More warps give the
  scheduler something to run while others wait. The original uses 166 registers and 41 KB of shared memory, and each
  of those alone limits it to 3 blocks (12 warps) per SM, which is 18.75% occupancy.
- **Warp cycles per issued instruction:** how long a warp waits, on average, between its instructions. The rows
  under it split that wait by reason:
  - *global loads* (long scoreboard): waiting for data from L2 or DRAM.
  - *global-memory queue full* (lg throttle): too many global-memory instructions in flight, typically atomics
    piling up.
  - *barrier:* waiting at `tl.debug_barrier()` for the rest of the block.
  - *tensor/math pipe busy:* the good kind of stall; the hardware is working.
  - *shared memory:* tensor-core operands pass through shared memory.
- **Atomic sectors per request:** 4 means perfectly coalesced (32 threads × 4 bytes = 128 bytes = 4 sectors). Higher
  means each atomic instruction is spread over more sectors. The count includes the `cnt` adds, which are a small
  share.

What to compare:
- original vs iter06 on `normal`: atomic sectors per request, lg throttle, duration.
- Each kernel on `synthetic` vs `normal`: real keys share centroids within a tile (9–23 distinct per 128 keys,
  against about 124 for random keys), so check whether the atomic stalls grow.
- int8 vs iter06: the int8 tensor pipe, and whether int8 still waits on the same things.

Takes about 5 minutes. Counters that this ncu version doesn't have are skipped and listed.

In [ ]:
import csv, subprocess
import pandas as pd

LABELS = {
    "gpu__time_duration.sum": "Duration (us, clocks locked)",
    "sm__throughput.avg.pct_of_peak_sustained_elapsed": "Compute (SM) throughput %",
    "gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed": "Memory throughput %",
    "l1tex__throughput.avg.pct_of_peak_sustained_active": "L1/TEX throughput %",
    "lts__throughput.avg.pct_of_peak_sustained_elapsed": "L2 throughput %",
    "dram__throughput.avg.pct_of_peak_sustained_elapsed": "DRAM throughput %",
    "sm__pipe_tensor_op_hmma_cycles_active.avg.pct_of_peak_sustained_active": "Tensor pipe, bf16 %",
    "sm__pipe_tensor_op_imma_cycles_active.avg.pct_of_peak_sustained_active": "Tensor pipe, int8 %",
    "sm__pipe_tensor_cycles_active.avg.pct_of_peak_sustained_active": "Tensor pipe, any type %",
    "launch__registers_per_thread": "Registers per thread",
    "launch__occupancy_limit_registers": "Blocks/SM allowed by registers",
    "launch__occupancy_limit_shared_mem": "Blocks/SM allowed by shared memory",
    "sm__warps_active.avg.pct_of_peak_sustained_active": "Achieved occupancy %",
    "smsp__average_warp_latency_per_inst_issued.ratio": "Warp cycles per issued instruction",
    "smsp__average_warps_issue_stalled_long_scoreboard_per_issue_active.ratio": "  global loads (long scoreboard)",
    "smsp__average_warps_issue_stalled_lg_throttle_per_issue_active.ratio": "  global-memory queue full (lg throttle)",
    "smsp__average_warps_issue_stalled_barrier_per_issue_active.ratio": "  barrier",
    "smsp__average_warps_issue_stalled_math_pipe_throttle_per_issue_active.ratio": "  tensor/math pipe busy",
    "smsp__average_warps_issue_stalled_mio_throttle_per_issue_active.ratio": "  shared-memory queue full (mio throttle)",
    "smsp__average_warps_issue_stalled_short_scoreboard_per_issue_active.ratio": "  shared-memory results (short scoreboard)",
    "smsp__average_warps_issue_stalled_wait_per_issue_active.ratio": "  fixed-latency wait",
    "l1tex__t_requests_pipe_lsu_mem_global_op_red.sum": "red requests",
    "l1tex__t_sectors_pipe_lsu_mem_global_op_red.sum": "red sectors",
    "l1tex__t_requests_pipe_lsu_mem_global_op_atom.sum": "atom requests",
    "l1tex__t_sectors_pipe_lsu_mem_global_op_atom.sum": "atom sectors",
}
SAFE = list(LABELS)[:6] + [m for m in LABELS if m.startswith(("launch__", "sm__warps_active"))]
known = {l.split()[0] for l in subprocess.run(["ncu", "--query-metrics"], capture_output=True, text=True).stdout.splitlines() if l.strip()}
metrics = [m for m in LABELS if m in SAFE or m.split(".")[0] in known]
print("skipped (not available):", [m for m in LABELS if m not in metrics] or "none")

def num(v):
    try:
        return float(v.replace(",", ""))
    except ValueError:
        return float("nan")

def profile(kernel, which, metrics):
    """One column of the table, or (None, ncu's output) if ncu failed."""
    cmd = ["ncu", "--profile-from-start", "off", "--csv", "--page", "raw", "--metrics", ",".join(metrics),
           "python", "prof.py", kernel, which]
    out = subprocess.run(cmd, capture_output=True, text=True, cwd="/content")
    lines = out.stdout.splitlines()
    start = next((i for i, l in enumerate(lines) if l.startswith('"ID"')), None)
    if start is None:
        return None, out.stdout[-1500:] + out.stderr[-1500:]
    rows = [r for r in csv.reader(lines[start:])]
    header = rows[0]
    units = next((dict(zip(header, r)) for r in rows[1:] if len(r) == len(header) and not r[0].strip()), {})
    launches = [dict(zip(header, r)) for r in rows[1:] if len(r) == len(header) and r[0].strip().isdigit()]
    main = max(launches, key=lambda d: num(d.get("gpu__time_duration.sum", "nan")))
    col = {LABELS[m]: num(main.get(m, "nan")) for m in metrics}
    scale = {"ns": 1e-3, "nsecond": 1e-3, "us": 1.0, "usecond": 1.0, "ms": 1e3, "msecond": 1e3}[units.get("gpu__time_duration.sum", "ns")]
    col["Duration (us, clocks locked)"] *= scale
    req = sum(col.get(k, 0) for k in ["red requests", "atom requests"])
    sec = sum(col.get(k, 0) for k in ["red sectors", "atom sectors"])
    for k in ["red requests", "atom requests", "red sectors", "atom sectors"]:
        col.pop(k, None)
    if req:
        col["Atomic requests (warp instructions)"] = req
        col["Atomic sectors per request"] = sec / req
    return col, None

cols = {}
for kernel in ["original", "iter06", "int8", "phase2"]:
    for which in ["synthetic", "normal"]:
        col, err = profile(kernel, which, metrics)
        if col is None and metrics != SAFE:
            print("ncu rejected a counter, retrying with the basic set. ncu said:\n" + err)
            metrics = SAFE
            col, err = profile(kernel, which, metrics)
        if col is None:
            raise RuntimeError(err)
        cols[f"{kernel}/{which}"] = col
        print("profiled", kernel, which)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 20)
df = pd.DataFrame(cols)
fmt = lambda v: "-" if v != v else (f"{v:,.0f}" if abs(v) >= 100 else f"{v:.3g}")
print((df.map(fmt) if hasattr(df, "map") else df.applymap(fmt)).to_string())

**Full Nsight report for one kernel.** Set `KERNEL` (original | iter06 | int8 | phase2) and `INPUT`
(synthetic | normal). The `OPT` / `INF` lines are Nsight's own suggestions.

In [ ]:
KERNEL, INPUT = "original", "normal"
!ncu --profile-from-start off --section SpeedOfLight --section ComputeWorkloadAnalysis --section MemoryWorkloadAnalysis --section Occupancy --section WarpStateStats --section LaunchStats python prof.py {KERNEL} {INPUT}

## 7. Checklist for the bullet

| Claim | Where to check | Earlier result |
|---|---|---|
| iter06 about 1.36× on real keys | section 4, per-row summary | old files' 14 ordinary rows: 1.35–1.38× |
| iter06 passes the strict test on real and held-out keys | section 3 | emulated worst cos gap 1.9e-4 |
| a third pass from iter06 found nothing faster | `kernel_agent.ipynb` step 6 log | 6 iterations, none better |
| int8 1.74× on synthetic keys | section 4, whole batches | 1.737× |
| int8 passes phase 1's test on synthetic keys, fails the strict test | section 3 | emulated: fails 3 of 16 real rows and synthetic |
| phase 2 is slower than iter06 on real keys | section 4 | old ordinary rows 1.25–1.28× |
| which change buys what | section 4, ablation | (new here) |
| clock drift between processes | section 5 | ~18% in the first runs |

Draft bullets (update the numbers from your outputs):
- *Optimized the Triton k-means assignment kernel from my sparse-attention paper (Double-P, arXiv 2602.05191) with a
  Claude-driven agent loop guided by Nsight Compute: **1.36× faster on real Qwen2.5-1.5B attention keys** (coalesced
  atomic adds, packed score/index argmax). Assignment error is bounded at 2^-12 in cosine similarity (16× finer than
  the bf16 inputs) and validated on held-out keys.*
- *Built the evaluation harness that made the results trustworthy: same-process interleaved timing, a real-key
  correctness gate with held-out keys, and a repeat-call check. It **rejected an int8 kernel (1.74×) that passed
  synthetic tests but picked wrong centroids**. Traced an apparent 2.4× to benchmark keys that skipped the
  pipeline's mean-centering, rebuilt them to match production, and confirmed with a further agent pass that the
  1.36× kernel was the best found.*